## Time Series Analysis and Multi Linear Regression Modeling
The model will be trained using pandas and scikit-learn.
The model will be trained from data found at https://www.kaggle.com/datasets/arashnic/cinema-ticket

I am running this through VS Code, using a docker container. You may also use the `ipynb` file in other Jupyter Notebook style setups. Consult Jupyter Notebook for options.

**To download the data, run this cell.**

Running this cell will download the data, if you are running it in the docker container. If not, you will need to navigate to the `KAGGLE_DATA_URL` and download the data manually.

In [1]:
import os
from data import download_kaggle_dataset
KAGGLE_DATA_URL = "https://www.kaggle.com/datasets/arashnic/cinema-ticket"
DATA_PATH = os.path.join(os.getcwd(), "data", "time_series_analysis")
download_kaggle_dataset(KAGGLE_DATA_URL, DATA_PATH)

/workspaces/MS365/src/data/time_series_analysis contains data. Delete the file(s) if you want to download again.


/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Import the necessary python packages**

Import `pandas`, `statsmodels.tsa.seasonal.seasonal_decompose`, `sklearn.linear_model.LinearRegression`, `sklearn.metrics.mean_absolute_error`, `sklearn.metrics.mean_squared_error`, and `matplotlib.pyplot`. Typically, packages such as `pandas`, `numpy`, and `matplotlib.pyplot` are imported with an allias. I will not be following that strategy here. 

The default size of the plots from `matplotlib.pyplot` is 6.4 inches by 4.8 inches ([width by height](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.figure.html)). I wll be setting them to 20 inches by 5 inches. This will create a more readable output. Depending on your screen size, you may want to change this for your own use. 

By default, `pandas` will truncate datasets with a lot of rows and a lot of columns. You can alter this functionality with the `set_option()` function. I have set it to show all possible columns. This could result in long run times for cells where you are displaying the data, if there are many columns to display. This will be expected behavior for this analysis.

If you are running the docker container or if you are using [Google Colab](https://colab.research.google.com/), the `pip install` has already been done. If not, then please consult your jupyter notebook environment docs for how to install the needed packages.

In [4]:
import pandas
import matplotlib.pyplot
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
import matplotlib.pyplot

matplotlib.pyplot.rcParams["figure.figsize"] = [20, 5]
pandas.set_option("display.max_columns", None)

**Import and Review Data**

Import the data downloaded from the Kaggle site. The file name is `cinemaTicket_Ref.csv`. Import it into `df` using `pandas.read_csv()`. The file is a typical CSV file, separated by commas.

Display the top rows of the dataframe using the `head()` method. The default value is `n=5`. Change the value if you want to see more or less rows. The example shows the top 15 rows of data.

In [3]:
df = pandas.read_csv(os.path.join(DATA_PATH, "cinemaTicket_Ref.csv"))
df.head(n=15)

,film_code,cinema_code,total_sales,tickets_sold,tickets_out,show_time,occu_perc,ticket_price,ticket_use,capacity,date,month,quarter,day
0,1492,304,3900000,26,0,4,4.26,150000.000000,26,610.328638,2018-05-05,5,2,5
1,1492,352,3360000,42,0,5,8.08,80000.000000,42,519.801980,2018-05-05,5,2,5
2,1492,489,2560000,32,0,4,20.00,80000.000000,32,160.000000,2018-05-05,5,2,5
3,1492,429,1200000,12,0,1,11.01,100000.000000,12,108.991826,2018-05-05,5,2,5
4,1492,524,1200000,15,0,3,16.67,80000.000000,15,89.982004,2018-05-05,5,2,5
5,1492,71,1050000,7,0,3,0.98,150000.000000,7,714.285714,2018-05-05,5,2,5
6,1492,163,1020000,10,0,3,7.69,102000.000000,10,130.039012,2018-05-05,5,2,5
7,1492,450,750000,5,0,3,1.57,150000.000000,5,318.471338,2018-05-05,5,2,5
8,1492,51,750000,11,0,2,0.95,68181.818182,11,1157.894737,2018-05-05,5,2,5
9,1492,522,600000,4,0,3,1.55,150000.000000,4,258.064516,2018-05-05,5,2,5


The top 15 rows of data provide a good example of the data and the potential datatypes. All the values are numerical, with the exception of `date`, which is a date column. This will need to be a `datetime` data type. 

It is also necessary to check for missing data and to verify the overall statistics of each column. Running the [`describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) will show the counts, the mean, the maximum value, the minimum value, and other statistics of the numerical columns. If the `include="all"` parameter is included, then the counts for all columns, including those that are not numerical will be included.

Looking at all the columns, the columns `capacity` and `occu_perc` have a lower count than the other columns. The count will only count values that are not null. This suggests that `capacity` and `occu_perc` contain some missing values. All other columns seem to have all their data points, which is 142,524 rows of data.

In [5]:
df.describe(include="all")

,film_code,cinema_code,total_sales,tickets_sold,tickets_out,show_time,occu_perc,ticket_price,ticket_use,capacity,date,month,quarter,day
count,142524.000000,142524.000000,1.425240e+05,142524.000000,142524.000000,142524.000000,142399.000000,142524.000000,142524.000000,142399.000000,142524,142524.000000,142524.000000,142524.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,234,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-05-15,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,790,NaN,NaN,NaN
mean,1518.985111,320.378427,1.234728e+07,140.137570,0.237413,3.932103,19.965986,81234.599886,139.900157,854.723605,NaN,6.776852,2.634721,16.112585
std,36.184450,159.701229,3.065486e+07,279.758733,2.923206,3.056276,22.653445,33236.599278,279.564935,953.118103,NaN,2.195843,0.809692,8.949471
min,1471.000000,32.000000,2.000000e+04,1.000000,0.000000,1.000000,0.000000,483.870968,-219.000000,-2.000000,NaN,2.000000,1.000000,1.000000
25%,1485.000000,181.000000,1.260000e+06,18.000000,0.000000,2.000000,3.750000,60000.000000,18.000000,276.994486,NaN,5.000000,2.000000,8.000000
50%,1498.000000,324.000000,3.720000e+06,50.000000,0.000000,3.000000,10.350000,79454.235185,50.000000,525.714286,NaN,7.000000,3.000000,16.000000
75%,1556.000000,474.000000,1.110000e+07,143.000000,0.000000,5.000000,28.210000,100000.000000,143.000000,1038.961039,NaN,9.000000,3.000000,24.000000


Review the `dtypes` of each column in the dataframe. The column `date` is an `object` data type. It needs to be a `datetime` data type so it can be used in the time series analysis. The other columns are fine, for now. The time series analysis will be a Multiple Linear Regression analysis, which will require numerical data.

In [10]:
df.dtypes

film_code                int64
cinema_code              int64
total_sales              int64
tickets_sold             int64
tickets_out              int64
show_time                int64
occu_perc              float64
ticket_price           float64
ticket_use               int64
capacity               float64
date            datetime64[ns]
month                    int64
quarter                  int64
day                      int64
dtype: object

Convert the `date` column to a `datetime` data type using the (`to_datetime()`)[https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html] method from `pandas`. There are many parameters that can be used with this method. For the purpose of this change, the default parameters are sufficient. Typing `df.dtypes` again, at the end, will show that the `date` column is now a `datetime` data type.

In [11]:
df.date = pandas.to_datetime(df.date)

df.dtypes

film_code                int64
cinema_code              int64
total_sales              int64
tickets_sold             int64
tickets_out              int64
show_time                int64
occu_perc              float64
ticket_price           float64
ticket_use               int64
capacity               float64
date            datetime64[ns]
month                    int64
quarter                  int64
day                      int64
dtype: object

The columns `capacity` and `occu_perc` contains 142,399 non-null values. The other columns contain 142,524 non-null values. This means that those columns contains 125 null values. The missing values will cause problems with the statistical analysis, so the values either need to be dropped or they need to be imputed. It is possible to run the pandas [`dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) function and the missing values would be removed from the dataset. But, if the missing values are dropped, there is the possibility of up to 250 rows of data are lost, since it is uncertain if the missing values are on the same rows of data. The second option is imputation or [replacing the missing value with estimates](https://www.analyticsvidhya.com/blog/2021/10/handling-missing-value/). The imputation method will be used here because I don't want to lose out on any of the data. If you have a really small dataset, then imputation will likely be your best strategy. If you have a very large dataset (think millions or billions of rows of data), then imputation may not be necessary, since there are so many rows of data.

To impute the data, use the pandas [`fillna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) function to replace all missing values with the strategy of your choice. There are many kinds of imputation. Filling in the missing values with the mean/median/mode tends to be the simplest. It is not necessarily the best, but will work for now. The first parameter of the `fillna()` function is the value you want to use to replace the missing data. You can either type in the specific value you want, or you can use the desired function on the column with the missing values and have it calculate the number before it changes all the missing values. I like using the mean-value because this means that the new values will not alter the current average for that column. However, this could be the wrong strategy if the data are not normally distributed or there are other such statistical anomalies.

*Hint: Using the `inplace=True` parameter may cause a warning about setting a value on a copy. To avoid that error, assign the values back to the column of the dataframe.*